# Embedding space visualization
This notebook tests t-SNE and UMAP algorithms to plot the embeddings and understand if clusters are formed depending on features such as type, style or century of the painting.

### 0. Import libraries and data

In [ ]:
import numpy as np
import polars as pl
import plotly.express as px
from umap import UMAP
from sklearn.manifold import TSNE
from plotly.subplots import make_subplots

INPUT_PATH_EMBEDDINGS = "../../data/embeddings/"

In [ ]:
projected_embeddings = (
    pl.read_json(
        f"{INPUT_PATH_EMBEDDINGS}baseline_embeddings_test_projections_text_embedding_enhanced_features.json",
        infer_schema_length=1000,
    )
    .sort("painting_id")
    .with_row_index()
    .with_columns((pl.col("year") // 100 + 1).alias("century"))
    .with_columns(
        pl.col("coarse_type").map_elements(
            lambda x: x if x != "history" else "historical", return_dtype=pl.String
        )
    )
)
probabilities = projected_embeddings["probability"]
projected_embeddings = (
    projected_embeddings.sort("probability", descending=True)
    .unique(subset=["object_description"], keep="first")
    .sort("index")
    .drop("index")
    .with_row_index()
)

clip_embeddings = (
    pl.read_json(
        f"{INPUT_PATH_EMBEDDINGS}clip_embeddings_test_clip_full_1e_6_diff_lr_not_frozen_features.json",
        infer_schema_length=1000,
    )
    .with_columns((pl.col("year") // 100 + 1).alias("century"))
    .with_columns(
        pl.col("coarse_type").map_elements(
            lambda x: x if x != "history" else "historical", return_dtype=pl.String
        )
    )
).with_columns(pl.Series(probabilities).alias("probability"))
clip_embeddings = (
    clip_embeddings.sort("probability", descending=True)
    .unique(subset=["object_description"], keep="first")
    .sort("index")
    .drop("index")
    .with_row_index()
)

In [ ]:
projected_embeddings

### 1. Plot the embeddings

In [ ]:
def project_embeddings(
    complete_input_embeddings,
    modality,
    feature_name,
    viz_algorithm="t-SNE",
    viz_algorithm_param=30,
    top_feature_values=10,
):
    feature_values = (
        complete_input_embeddings.filter(pl.col(feature_name).is_not_null())[feature_name]
        .value_counts()
        .sort("count", descending=True)[:top_feature_values][feature_name]
        .to_list()
    )
    input_embeddings = complete_input_embeddings.filter(pl.col(feature_name).is_in(feature_values))

    if modality == "visual":
        embeddings = np.stack(input_embeddings["embedding_object_image"].to_numpy())
    elif modality == "textual":
        embeddings = np.stack(input_embeddings["text_embedding_enhanced"].to_numpy())

    if viz_algorithm == "t-SNE":
        tsne = TSNE(
            n_components=2,
            perplexity=viz_algorithm_param,
            random_state=42,
            metric="cosine",
            method="exact",
        )
        embeddings_2d = tsne.fit_transform(embeddings)
    elif viz_algorithm == "UMAP":
        umap = UMAP(
            n_neighbors=viz_algorithm_param,
            min_dist=0.1,
            n_components=2,
            random_state=42,
            n_jobs=1,
            metric="cosine",
        )
        embeddings_2d = umap.fit_transform(embeddings)

    embeddings_2d = (
        pl.DataFrame(embeddings_2d)
        .with_columns(pl.Series(input_embeddings[feature_name]).cast(pl.String).alias(feature_name))
        .with_columns(
            pl.Series(input_embeddings["object_description"])
            .cast(pl.String)
            .alias("object_description")
        )
        .with_columns(pl.Series(input_embeddings["label"]).cast(pl.String).alias("label"))
    )

    return embeddings_2d

In [ ]:
def plot_embeddings(
    projected_embeddings_clip,
    projected_embeddings_proposed_method,
    modality,
    feature_name,
    viz_algorithm="t-SNE",
    font_size=18,
):
    symbol_sequence = [
        "circle", "square", "diamond", "cross", "x",
        "triangle-up", "triangle-down", "star", "triangle-right", "triangle-left"
    ]
    color_sequence = px.colors.qualitative.Bold

    
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Projected Grounding DINO Embeddings", "CLIP Embeddings"),
        shared_xaxes=True, 
        shared_yaxes=True,  
        horizontal_spacing=0.02,
    )

    fig1_px = px.scatter(
        projected_embeddings_proposed_method,
        x="column_0",
        y="column_1",
        color=feature_name,
        symbol=feature_name,
        color_discrete_sequence=color_sequence,
        symbol_sequence=symbol_sequence,
        hover_data=[feature_name, "object_description", "label"],
    )


    fig2_px = px.scatter(
        projected_embeddings_clip,
        x="column_0",
        y="column_1",
        color=feature_name,
        symbol=feature_name,
        color_discrete_sequence=color_sequence,
        symbol_sequence=symbol_sequence,
        hover_data=[feature_name, "object_description", "label"],
    )

    for trace in fig1_px.data:
        fig.add_trace(trace, row=1, col=1)

    for trace in fig2_px.data:
        trace.showlegend = False
        fig.add_trace(trace, row=1, col=2)

    for annotation in fig.layout.annotations:
        annotation.font.size = font_size + 2  

    fig.update_layout(
        yaxis_title="t-SNE Dimension 2",
        legend_title="Object Label",
        font=dict(size=font_size),
        title_font_size=font_size + 4, 
        xaxis=dict(title_font_size=font_size, tickfont_size=font_size),
        yaxis=dict(title_font_size=font_size, tickfont_size=font_size),
        legend=dict(font_size=font_size - 2),
        width=1600,
        height=700,
    )

    fig.update_xaxes(
        zeroline=True,
        zerolinewidth=1, 
    )

    fig.update_yaxes(
        zeroline=True,
        zerolinewidth=1,
    )

    fig.add_annotation(
        x=0.5,
        y=-0.15,
        xref="paper", 
        yref="paper",
        text="t-SNE Dimension 1",
        showarrow=False,
        font=dict(size=font_size),
        align="center",
    )

    fig.update_traces(marker=dict(size=10))
    fig.show()
    fig.write_image(f"embeddings_2d_{modality}_{feature_name}.png", scale=2)

In [ ]:
modality = "textual"
feature = "label"
perplexity = 10

projected_embeddings_clip = project_embeddings(
    clip_embeddings.clone(), modality, feature, "t-SNE", perplexity
)

projected_embeddings_proposed_method = project_embeddings(
    projected_embeddings.clone(), modality, feature, "t-SNE", perplexity
)

plot_embeddings(
    projected_embeddings_clip,
    projected_embeddings_proposed_method,
    modality,
    feature,
    viz_algorithm="t-SNE",
    font_size=18,
)